Question 3

In [4]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [5]:
TARGET_COL = "price"
DROP_COLS = ["id", "date", "zipcode", "Unnamed: 0"]

def load_prepare(path: str):
    df = pd.read_csv(path)

    y = (df[TARGET_COL].astype(float) / 1000.0).to_numpy()
    X = df.drop(columns=[TARGET_COL] +[c for c in DROP_COLS if c in df.columns], errors="ignore")

    #only keep numeric columns
    X = X.select_dtypes(include=[np.number])
    return X, y

def fit_closed_form(X: np.ndarray, y: np.ndarray):
    # w = pinv(X_aug) @ y where X_aug = [1, X]
    # w[0] = intercept, w[1:] = coefficients
    X_aug = np.column_stack([np.ones(X.shape[0]), X])
    w = np.linalg.pinv(X_aug) @ y
    return w

def predict_closed_form(X: np.ndarray, w: np.ndarray):
    X_aug = np.column_stack([np.ones(X.shape[0]), X])
    return X_aug @ w

In [6]:
train_path = "train.csv"
test_path = "test.csv"

X_train_df, y_train = load_prepare(train_path)
X_test_df, y_test = load_prepare(test_path)

X_train_df, X_test_df = X_train_df.align(X_test_df, join="inner", axis=1)
feature_names = X_train_df.columns.tolist()
imputer = SimpleImputer(strategy = "median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train_df))
X_test = scaler.transform(imputer.transform(X_test_df))

w = fit_closed_form(X_train, y_train)
train_pred_cf = predict_closed_form(X_train , w)
test_pred_cf = predict_closed_form(X_test, w)

train_mse_cf = mean_squared_error(y_train, train_pred_cf)
test_mse_cf = mean_squared_error(y_test, test_pred_cf)
train_r2_cf = r2_score(y_train, train_pred_cf)
test_r2_cf = r2_score(y_test, test_pred_cf)

print("Closed-form results: ")
print(f"Train MSE: {train_mse_cf}   R2: {train_r2_cf}")
print(f"Test MSE: {test_mse_cf}   R2: {test_r2_cf}")
print(f"Intercept (w0): {w[0]}")
print("\nCoefficients (standardized X):")
for name, coef in sorted(zip(feature_names, w[1:]), key=lambda t: abs(t[1]), reverse=True):
    print(f"{name:>15}  {coef: .6f}")

Closed-form results: 
Train MSE: 31486.167775794882   R2: 0.7265334318706018
Test MSE: 57628.154705670415   R2: 0.6543560876120953
Intercept (w0): 520.4148340000008

Coefficients (standardized X):
          grade   92.231475
            lat   78.375737
       yr_built  -67.643117
     waterfront   63.742900
    sqft_living   56.748837
     sqft_above   48.290089
           view   48.200109
  sqft_living15   45.577658
  sqft_basement   27.137032
      bathrooms   18.527633
   yr_renovated   17.271380
      condition   12.964269
     sqft_lot15  -12.930091
       bedrooms  -12.521962
       sqft_lot   10.881868
         floors   8.043721
           long  -1.035203


In [7]:
# sklearn model for comparison
sk = LinearRegression()
sk.fit(X_train , y_train)

train_pred_sk = sk.predict(X_train)
test_pred_sk = sk.predict(X_test)

train_mse_sk = mean_squared_error(y_train, train_pred_sk)
test_mse_sk = mean_squared_error(y_test, test_pred_sk)
train_r2_sk = r2_score(y_train, train_pred_sk)
test_r2_sk = r2_score(y_test, test_pred_sk)

print("\nsklearn results: ")
print(f"Train MSE: {train_mse_sk}   R2: {train_r2_sk}")
print(f"Test  MSE: {test_mse_sk}   R2: {test_r2_sk}")


sklearn results: 
Train MSE: 31486.167775794882   R2: 0.7265334318706018
Test  MSE: 57628.154705670415   R2: 0.6543560876120953


In [8]:
#compare parameter differences
max_coef_diff = np.max(np.abs(w[1:] - sk.coef_))
intercept_diff = abs(w[0] - sk.intercept_)

print("\nParameter Differences: ")
print("Max |coef_closed_form - coef_sklearn|:", max_coef_diff)
print("|intercept_closed_form - intercept_sklearn|:", intercept_diff)


Parameter Differences: 
Max |coef_closed_form - coef_sklearn|: 2.7711166694643907e-13
|intercept_closed_form - intercept_sklearn|: 1.1368683772161603e-13


My closed form multiple linear regression implementation produces essentially the same model as the package based linear regression from problem 2. Using the same preprocessing, the closed form model gets Train MSE = 31486.1678 and Train R-squared = 0.7265, and on the test set Test MSE = 57628.1547 and Test R-squared = 0.6544. These values match the sklearn results from problem 2 (Train MSE = 31486.1678, Train R-squared = 0.7265, Test MSE = 57628.1547, Test R-squared = 0.6544). This is expected because both approaches solve the same ordinary least squares objective, just using different routines.